# Single shooting through a CartPole rollout
## From $\partial f$ to $\nabla J$ — tip the pole to $\theta^\star$, plan $U$ or co-design $(U, p)$

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/sandbox/scratch/cartpole_rollout_gradients.ipynb)

One pure dynamics map $\dot{x}=f(x,u;p)$ is enough to *differentiate an entire plan*. Compose one integration step into a rollout, score the final state, and automatic differentiation returns the gradient of that cost — no hand-written adjoints.

**This notebook.** Tip a 2-dof cart–pole from hanging toward a target angle ($\theta^\star=1$) by *single shooting*:
1. optimize the force sequence $U$ alone;
2. co-optimize $U$ with physical parameters $p=(\ell_{\mathrm{cg}}, m_1, m_2)$.

Companion script: [`trajopt_cartpole_rollout_gradients.py`](../../demos/trajopt/trajopt_cartpole_rollout_gradients.py) · related: [`showcase/jax.ipynb`](../../learn/intro/showcase_jax.ipynb) §7.


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone -b main https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## The math chain

### 1. Continuous dynamics

The catalog plant `JaxCartPole` is the usual underactuated cart–pole. State $x=[\,x_{\mathrm{cart}},\,\theta,\,\dot x,\,\dot\theta\,]$, force $u=F$, parameters $p$:

$$
\dot{x} = f(x, u;\, p).
$$

Because $f$ is a pure function of $(x,u,p)$, its Jacobians $\partial f/\partial x$, $\partial f/\partial u$, and $\partial f/\partial p$ exist and are available by autodiff.

### 2. One discrete step

An RK4 step of size $\Delta t$ is itself a pure map

$$
x_{k+1} = \Phi\!\big(x_k,\, u_k;\, p\big).
$$

Differentiating the step is the chain rule through $f$ at the four RK4 stages — still automatic.

### 3. Rollout = single shooting

Open-loop plan $U=(u_0,\ldots,u_{N-1})$. The $N$-step composition

$$
x_N(U, p)
\;=\;
\Phi\big(\cdots\,\Phi\big(\Phi(x_0, u_0; p),\, u_1; p\big)\cdots,\, u_{N-1}; p\big)
$$

is *single shooting*: one free initial state, one decision sequence, one simulated trajectory. No collocation defects — the dynamics are enforced by construction.

### 4. Terminal cost

Land the pole on $\theta^\star$, with light cart / rate / effort penalties:

$$
J_{\mathrm{task}}(U, p)
\;=\;
5\big(\theta_N - \theta^\star\big)^2
\;+\;
\varepsilon_x\, x_{\mathrm{cart},N}^2
\;+\;
\varepsilon_v\|\dot q_N\|^2
\;+\;
\varepsilon_u\sum_{k=0}^{N-1} u_k^2.
$$

### 5. Gradients by the chain rule

$$
\nabla_U J
\;=\;
\frac{\partial J}{\partial x_N}\,
\frac{\partial x_N}{\partial U},
\qquad
\nabla_p J
\;=\;
\frac{\partial J}{\partial x_N}\,
\frac{\partial x_N}{\partial p}.
$$

The rollout sensitivities $\partial x_N/\partial U$ and $\partial x_N/\partial p$ are exactly what `jax.value_and_grad` builds by differentiating through $\Phi\circ\cdots\circ\Phi$.

### 6. Gradient descent

$$
U \leftarrow \Pi_{[-u_{\max},u_{\max}]}\!\big(U - \alpha\,\nabla_U J\big),
\qquad
p \leftarrow \Pi_{\mathcal{P}}\!\big(p - \beta\,\nabla_p J\big)
$$

with box projections on force and on admissible $(\ell_{\mathrm{cg}}, m_1, m_2)$. Same $J_{\mathrm{task}}$ in both experiments below; (b) adds a soft stay-near-nominal regularizer on $p$.


## Swing-up task

| | |
| --- | --- |
| Plant | `JaxCartPole` (2 dof, 1 force) |
| $x_0$ | hanging, tiny tip: $\theta_0=0.05$ |
| $\theta^\star$ | $1.0\,\mathrm{rad}$ |
| Horizon | $N=50$, $\Delta t=0.05\,\mathrm{s}$ ($2.5\,\mathrm{s}$) |
| (a) | $\min_U\, J_{\mathrm{task}}(U, p_{\mathrm{nom}})$ |
| (b) | $\min_{U,p}\, J_{\mathrm{task}}(U,p) + w_p\|p-p_{\mathrm{nom}}\|^2$ |


### Setup — run once (ceremony)

Compiles the plant, defines $\Phi$ / $J$ / GD, and helpers that wrap results as minilink `Trajectory` objects. **Skip reading on a first pass**; the story cells below only call these.


In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.backends import configure_jax
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.cartpole import JaxCartPole

configure_jax(enable_x64=True)

rk4_step_p = JaxCartPole().compile(backend="jax").rk4_step_p

H, dt = 50, 0.05
x0 = jnp.array([0.0, 0.05, 0.0, 0.0])
theta_star = 1.0  # float(np.pi)  # upright
p_nom = jnp.array([0.5, 1.0, 0.1])  # (lcg, m1, m2)
p_lo = jnp.array([0.2, 0.4, 0.05])
p_hi = jnp.array([2.5, 3.0, 1.0])
w_p, u_bound = 1e-2, 10.0


def _params(p):
    return {"lcg": p[0], "m1": p[1], "m2": p[2], "gravity": 9.81}


def rollout(U, p):
    params = _params(p)

    def step(x, u_k):
        x_next = rk4_step_p(x, jnp.array([u_k]), 0.0, dt, params)
        return x_next, x_next

    return jax.lax.scan(step, x0, U)


def J_task(x_N, U):
    return (
        5.0 * (x_N[1] - theta_star) ** 2
        + 0.2 * x_N[0] ** 2
        + 0.5 * jnp.sum(x_N[2:] ** 2)
        + 2e-4 * jnp.sum(U**2)
    )


def as_trajectory(states, U):
    x = np.vstack([np.asarray(x0), np.asarray(states)])
    u = np.vstack([np.asarray(U).reshape(-1, 1), [[float(U[-1])]]])
    return Trajectory(t=dt * np.arange(x.shape[0]), x=x.T, u=u.T)


def plant_with(p, name):
    sys = JaxCartPole()
    sys.name = name
    for k, v in _params(p).items():
        sys.params[k] = float(v)
    sys.pole_length = max(1.5, 6.0 * float(p[0]))
    return sys


def pump_init():
    return 0.5 * jnp.sin(2.0 * jnp.pi * jnp.arange(H) / H)


def solve_U_only():
    def cost(U):
        x_N, _ = rollout(U, p_nom)
        return J_task(x_N, U)

    value_and_grad = jax.jit(jax.value_and_grad(cost))
    U = pump_init()
    for lr in (0.4, 0.2, 0.1, 0.05):
        for _ in range(1200):
            _, g = value_and_grad(U)
            U = jnp.clip(U - lr * g, -u_bound, u_bound)
    x_N, xs = rollout(U, p_nom)
    return dict(U=U, p=p_nom, x_N=x_N, xs=xs, J=float(cost(U)))


def solve_co():
    def cost(dec):
        U, p = dec
        x_N, _ = rollout(U, p)
        return J_task(x_N, U) + w_p * jnp.sum((p - p_nom) ** 2)

    value_and_grad = jax.jit(jax.value_and_grad(cost))
    U, p = pump_init(), p_nom
    for lr_u, lr_p in ((0.4, 0.03), (0.2, 0.015), (0.1, 0.008), (0.05, 0.003)):
        for _ in range(1200):
            _, (g_u, g_p) = value_and_grad((U, p))
            U = jnp.clip(U - lr_u * g_u, -u_bound, u_bound)
            p = jnp.clip(p - lr_p * g_p, p_lo, p_hi)
    x_N, xs = rollout(U, p)
    return dict(
        U=U, p=p, x_N=x_N, xs=xs, J=float(cost((U, p))), J_task=float(J_task(x_N, U))
    )


def summarize(tag, sol):
    x_N, U, p = sol["x_N"], np.asarray(sol["U"]), np.asarray(sol["p"])
    print(
        f"{tag}:  J={sol['J']:.3e}  "
        f"θ_N={float(x_N[1]):.3f} (★ {theta_star:.3f})  "
        f"u_rms={np.sqrt(np.mean(U**2)):.3f} N  "
        f"p=(ℓ={p[0]:.3f}, m1={p[1]:.3f}, m2={p[2]:.3f})"
    )


## (a) Single shooting on $U$ only

Hold $p=p_{\mathrm{nom}}$ fixed and descend on $\nabla_U J_{\mathrm{task}}$. Decision variable: the open-loop force sequence. Custom plots, then one catalog animation.


In [ ]:
sol_a = solve_U_only()
summarize("(a) U-only", sol_a)

t = dt * np.arange(H)
xs_a, U_a = np.asarray(sol_a["xs"]), np.asarray(sol_a["U"])

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(t, xs_a[:, 0], label=r"$x$")
ax[0].plot(t, xs_a[:, 1], label=r"$\theta$")
ax[0].axhline(theta_star, ls="--", c="k", lw=0.8, label=r"$\theta^\star$")
ax[0].set_xlabel("t [s]"); ax[0].legend(); ax[0].set_title("(a) state")
ax[1].step(t, U_a, where="post", c="C1")
ax[1].axhline(0, c="k", lw=0.5)
ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("F [N]"); ax[1].set_title(r"(a) $U^\star$")
fig.tight_layout(); plt.show()

sys_a = plant_with(sol_a["p"], "CartPole (a) U-only")
sys_a.animate(as_trajectory(sol_a["xs"], sol_a["U"]))


## (b) Co-optimize $U$ and $p$

Same $J_{\mathrm{task}}$, but the decision is now $(U, p)$. Designing the plant while planning typically reaches upright with a different effort profile — the chain rule through $\partial x_N/\partial p$ makes that tradeoff visible to GD.


In [ ]:
sol_b = solve_co()
summarize("(b) co-opt", sol_b)

xs_b, U_b, p_b = np.asarray(sol_b["xs"]), np.asarray(sol_b["U"]), np.asarray(sol_b["p"])

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].plot(t, xs_a[:, 1], ls=":", label="(a)")
ax[0].plot(t, xs_b[:, 1], label="(b)")
ax[0].axhline(theta_star, ls="--", c="k", lw=0.8, label=r"$\theta^\star$")
ax[0].set_xlabel("t [s]"); ax[0].set_ylabel(r"$\theta$"); ax[0].legend(); ax[0].set_title("pole angle")
ax[1].step(t, U_a, where="post", ls=":", c="C1", label="(a)")
ax[1].step(t, U_b, where="post", c="C1", label="(b)")
ax[1].axhline(0, c="k", lw=0.5)
ax[1].set_xlabel("t [s]"); ax[1].legend(); ax[1].set_title("force")
ax[2].bar(np.arange(3) - 0.15, np.asarray(p_nom), 0.3, label="nom", color="0.7")
ax[2].bar(np.arange(3) + 0.15, p_b, 0.3, label=r"$p^\star$", color="C2")
ax[2].set_xticks(np.arange(3), [r"$\ell_{\mathrm{cg}}$", r"$m_1$", r"$m_2$"])
ax[2].legend(); ax[2].set_title("params")
fig.tight_layout(); plt.show()

print(
    f"|θ_N-θ★|  (a) {abs(float(sol_a['x_N'][1]) - theta_star):.3e}   "
    f"(b) {abs(float(sol_b['x_N'][1]) - theta_star):.3e}   |  "
    f"u_rms  (a) {np.sqrt(np.mean(U_a**2)):.3f}   (b) {np.sqrt(np.mean(U_b**2)):.3f} N"
)

sys_b = plant_with(sol_b["p"], "CartPole (b) co-opt")
sys_b.animate(as_trajectory(sol_b["xs"], sol_b["U"]))


## Takeaway

| Layer | Object | What AD differentiates |
| --- | --- | --- |
| physics | $f(x,u;p)$ | $\partial f/\partial x,\;\partial f/\partial u,\;\partial f/\partial p$ |
| step | $\Phi(x,u;p)$ | one RK4 map |
| plan | $x_N(U,p)$ | single-shooting rollout |
| objective | $J(U,p)$ | $\nabla_U J$ and/or $\nabla_p J$ |

Write $f$ once. Single shooting + gradient descent does the rest — whether the decision lives in $U$, in $p$, or in both.
